# 🧠 Continual Pretraining of Mistral-7B on Telecom Domain Text

This notebook documents the **continual pretraining** of the [Mistral-7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) language model on our **domain-specific telecom dataset (text file)** derived from **cleaned ITU (International Telecommunication Union) standards documents**.

The goal is to adapt Mistral-7B to the **telecommunication domain**, allowing the model to better understand specialized terms, formal structure, and context found in ITU regulatory and technical documentation. This step is a **precursor to instruction fine-tuning**, where the model will be trained to follow instructions in a task-specific manner using prompts and responses.

---

### 📌 Project Highlights

- **Dataset**: Cleaned telecom-focused text corpus compiled from ITU standards documents.
- **Base Model**: [`mistralai/Mistral-7B-v0.3`](https://huggingface.co/mistralai/Mistral-7B-v0.1)
- **Approach**: Continual pretraining using Hugging Face's `transformers` library.
- **Purpose**: Domain adaptation before applying instruction fine-tuning.
- **Expected Outcome**: Improved performance on telecom-related tasks post-finetuning.

---


## 📦 Install Required Libraries

This section installs the necessary Python packages for continual pretraining of the Mistral-7B model:

- `peft`: For Parameter-Efficient Fine-Tuning techniques like LoRA.
- `accelerate`: From Hugging Face; manages multi-GPU or TPU training.
- `bitsandbytes`: Enables 8-bit and 4-bit quantization for memory-efficient training.
- `transformers`: Core library for working with pretrained language models.
- `datasets`: For loading and managing large-scale datasets.
- `GPUtil`: Helps in querying GPU availability and memory.


In [1]:
%%capture
# Install the required libraries silently (%%capture suppresses the output)

!pip install peft            # For parameter-efficient fine-tuning (e.g., LoRA, QLoRA)
!pip install accelerate      # accelerate fro managing multi-GPU or TPU training.
!pip install bitsandBytes    # Library for quantization (8-bit, 4-bit) to reduce memory usage
!pip install transformers    # Hugging Face Transformers library for model loading and training
!pip install datasets        # Hugging Face Datasets library for handling datasets
!pip install GPUtil          # Utility to check and manage GPU resources


## ⚙️ GPU Availability and Device Configuration

This section checks for the availability of a GPU and configures the CUDA device environment:

- Uses `GPUtil` to display current GPU utilization.
- Checks whether a CUDA-compatible GPU is available.
- If available, prints a confirmation; otherwise, sets the device to CPU.
- Sets environment variables to control CUDA device ordering and visibility.

In [2]:
import torch
import GPUtil
import os

# Display current GPU usage and memory stats
GPUtil.showUtilization()

# Check if a CUDA-compatible GPU is available
if torch.cuda.is_available():
    print("GPU is available")
else:
    device = torch.device("cpu")  # Fallback to CPU 
    print("GPU is not available, using CPU instead")

# Set environment variable to use CUDA device ordering based on PCI_BUS_ID
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# Make only GPU 0 visible to CUDA (i.e., restrict training to GPU 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

| ID | GPU | MEM |
------------------
|  0 |  0% |  0% |
GPU is available


## 🧱 Import Libraries and Model Utilities

This section imports the core libraries and utilities needed for:

- Loading the base model (`AutoModelForCausalLM`) and tokenizer.
- Applying quantization configuration (`BitsAndBytesConfig`) for efficient memory usage.
- Accessing datasets using `load_dataset`.
- Logging into the Hugging Face Hub (`notebook_login`) to download/upload models.
- Preparing the model for efficient fine-tuning using **LoRA** via `peft`.


In [3]:
# Hugging Face Transformers core module
import transformers

# for load tokenizer and causal language model (e.g., Mistral-7B)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# For logging in to Hugging Face Hub (e.g., for access to private models or pushing results)
from huggingface_hub import notebook_login

# Load datasets from the Hugging Face Datasets library
from datasets import load_dataset

# Tools for parameter-efficient fine-tuning (PEFT) using LoRA
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model


2025-11-22 19:33:34.024510: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763840014.204887      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763840014.255898      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

## 🔐 Authenticate with Hugging Face and Weights & Biases

This section securely retrieves and configures authentication credentials for:

- **Hugging Face Hub**: To access or push models (via `hf_api_key`).
- **Weights & Biases (wandb)**: For experiment tracking and visualizing training metrics.

The keys are securely retrieved using Kaggle’s `UserSecretsClient` (useful in notebooks running on Kaggle). After authentication:
- The Hugging Face client is logged in for model interaction.
- Weights & Biases is initialized for tracking the continual pretraining run.


In [4]:
# Securely access stored secrets using Kaggle's UserSecretsClient
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# Retrieve Hugging Face token and Weights & Biases token
hf_api_key = user_secrets.get_secret("HF token")
wandb_api_key = user_secrets.get_secret("wandb")

In [5]:
# Log in to Hugging Face Hub using the retrieved token
from huggingface_hub import login
login(token=hf_api_key)
print("Successfully logged into Hugging Face Hub!")

Successfully logged into Hugging Face Hub!


In [6]:
# Import Weights & Biases for experiment tracking
import wandb

# Authenticate to wandb using the retrieved API key
wandb.login(key=wandb_api_key)

# Initialize a new run in the specified wandb project
run = wandb.init(
    project="Cintinually Pre-trained Mistral-7B-Instruct-v0.3",  # Typo in 'Continually' left unchanged
    job_type="training",
    anonymous="allow"
)

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

## 🧠 Load Mistral-7B-Instruct with 4-Bit Quantization

In this step, we load the base model for continual pretraining:

- **Model**: [`mistralai/Mistral-7B-Instruct-v0.3`](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3)
  - A **7 billion parameter**, decoder-only transformer model.
  - Trained by [Mistral AI](https://mistral.ai) for **instruction following** (chat-style use cases).
  - It builds on the original Mistral-7B foundation model and is fine-tuned for better alignment and task following.
  - Suitable for use cases like QA, summarization, and dialog — even more so after domain adaptation.

We apply **4-bit quantization** using the `BitsAndBytesConfig` to reduce GPU memory usage:

- `load_in_4bit=True`: Enables loading model weights in 4-bit precision.
- `bnb_4bit_use_double_quant=True`: Uses two-stage quantization for better accuracy.
- `bnb_4bit_quant_type="nf4"`: Uses the **normal float 4 (NF4)** quantization scheme — optimal for LLMs.
- `bnb_4bit_compute_dtype=torch.bfloat16`: Computation is done in `bfloat16` for performance + stability.

Quantization is critical when working with large models like Mistral-7B on limited hardware (e.g., single GPU).


In [7]:
# Define the base model to be used for continual pretraining
base_model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# Configure 4-bit quantization using bitsandbytes
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,                        # Load model weights in 4-bit precision to save memory
    bnb_4bit_use_double_quant = True,           # Use double quantization (helps improve accuracy)
    bnb_4bit_quant_type = "nf4",                # Use the NF4 quantization format (optimized for transformers)
    bnb_4bit_compute_dtype = torch.bfloat16     # Use bfloat16 for computations (faster, stable)
)

# Load the pre-trained Mistral-7B-Instruct model with the specified quantization config
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config = bnb_config           # Apply quantization settings when loading the model
)


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

## 📝 Prepare Pubmed Dataset for Pretraining

This section loads the pubmed dataset corpus from huggingface.

### Key Steps:
- **Blank line removal**: Skips empty lines to avoid noise.
- **Chunking**: Groups every `N` lines (e.g., 10) into a single training sample.
  - This simulates longer text contexts while staying within memory limits.
  - You can increase `chunk_size` to make inputs more informative (if your system supports it).
- **Hugging Face Dataset**: Wraps the processed text into a format compatible with the Transformers training API.

This prepares a high-quality telecom-specific corpus for next-token prediction tasks.


In [8]:
from datasets import Dataset, load_dataset

ds = load_dataset("ccdv/pubmed-summarization", "section")

combined_article = "\n".join(ds["train"]["article"])

README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00005.parquet:   0%|          | 0.00/210M [00:00<?, ?B/s]

section/train-00001-of-00005.parquet:   0%|          | 0.00/208M [00:00<?, ?B/s]

section/train-00002-of-00005.parquet:   0%|          | 0.00/207M [00:00<?, ?B/s]

section/train-00003-of-00005.parquet:   0%|          | 0.00/211M [00:00<?, ?B/s]

...

section/validation-00000-of-00001.parque(…):   0%|          | 0.00/59.0M [00:00<?, ?B/s]

section/test-00000-of-00001.parquet:   0%|          | 0.00/58.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/119924 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6633 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6658 [00:00<?, ? examples/s]

In [9]:
def chunk_by_words(text, max_chars):
    words = text.split()
    chunks = []
    current = []

    current_len = 0

    for w in words:
        # If adding the next word exceeds chunk size → start new chunk
        if current_len + len(w) + 1 > max_chars:
            chunks.append(" ".join(current))
            current = [w]
            current_len = len(w)
        else:
            current.append(w)
            current_len += len(w) + 1

    # Final chunk
    if current:
        chunks.append(" ".join(current))

    return chunks


In [10]:
chunks = chunk_by_words(combined_article, 2000)

In [11]:
# Create a Hugging Face-compatible dataset with a 'text' column
import numpy as np
train_dataset = Dataset.from_dict({"text": [chunks[i] for i in np.random.randint(0, len(chunks), 5000)]})

In [12]:
train_dataset["text"][58]

'dissociations were estimated with the help of molecular - mechanical models , in which the tubulin monomer / dimer was the smallest unit . the major drawback of this approach is that tubulin energies are derived from the dynamic parameters of mt assembly and disassembly , which report on the thermodynamics of tubulin tubulin interactions only indirectly . in contrast , the afm - based dynamic force measurements provide a more straightforward experimental avenue , because in these experiments the protofilaments deformation and tubulin tubulin bond rupture events are recorded with high spatial and temporal resolution . however , due to the complexity of the multi - protofilament mt structure , the molecular interpretation of experimental force indentation spectra at the level of protein protein bonds is not trivial , as it requires the structure - based understanding of fine features of the experimental spectra . we have overcome this limitation by carrying out the dynamic force measure

In [13]:
train_dataset["text"][872]

'for 2 h. the dark - red reaction mixture was poured slowly over ice - cold water ( 400 ml ) and kept aside for 1 h to form a white precipitate . h nmr ( cdcl3 , 400 mhz ) in ppm : 2.92 ( m , 2h ) , 3.213.25 ( m , 2h ) , 7.36 ( s , 1h ) , 8.10 ( s , 1h ) . c nmr ( cdcl3 , 100 mhz ) in ppm : 26.8 , 39.1 , 129.1 , 129.8 , 130.3 , 130.7 , 138.1 , 141.8 , 192.2 . purity of 100% as determined by rp - hplc , tr = 17.33 min ( linear gradient system of 0100% b in a for 26 min ) . esi - ms calcd mw for c9h6cl2os , 233.11 ; found , m / z = 232.81 and 234.83 ( m + h ) . off - white solid ; yield 2.3 g ( 88% ) ; mp 112115 c . h nmr ( cdcl3 , 500 mhz ) in ppm : 2.87 ( m , 2h ) , 3.37 ( m , 2h ) , 7.427.45 ( m , 1h ) , 7.877.89 ( m , 1h ) . c nmr ( cdcl3 , 125 mhz ) in ppm : 26.0 , 37.8 , 126.4 , 128.5 , 128.7 , 131.4 , 137.5 , 144.3 , 192.9 . purity of 99.1% as determined by rp - hplc , tr = 16.88 min ( linear gradient system of 0100% b in a for 26 min ) . esi - ms calcd mw for c9h6cl2os , 233.11 ;

In [14]:
train_dataset["text"][226]

'tertiary intensive care center with 50 bed capacity . twelve infants who received intravenous colistin treatment for mdr gram - negative bacterial infections ( all isolated microorganisms were resistant to aminoglycosides , quinolones , penicillins , all cephalosporins and inhibitor combinations , monobactams , and carbapenems ) between january 2013 and february 2014 were retrospectively evaluated . the retrospective study was approved by the ethics committee of faculty of medicine of ataturk university . standard definitions of center for disease control and prevention ( cdc ) were used for nosocomial infection ( 12 ) . the definition of and diagnostic criteria for ventilator - associated pneumonia ( vap ) were identical with those of the cdc for infants less than 1 year of age ( 12 ) : the time of mechanical ventilation > 48 hours , new or persistent infiltrations on chest x - ray , worsening gas exchange and at least three of the following : ( a ) temperature instability with no ot

## 🔡 Tokenization of Text Data

This section initializes the tokenizer and tokenizes the telecom dataset for model training.

### Key Actions:
- **Load Tokenizer**: Loads the tokenizer from the base model (`mistralai/Mistral-7B-Instruct-v0.3`).
  - `use_fast=False`: Uses the slower but more customizable version.
  - `trust_remote_code=True`: Allows loading custom tokenizer logic (as required by Mistral).
  - `add_eos_token=True`: Appends an EOS (end-of-sequence) token automatically.

- **Pad Token Handling**: Ensures the tokenizer has a padding token. If missing, it uses the EOS token as a surrogate pad token.

- **Tokenization Loop**: Iterates over the dataset and tokenizes each sample individually using the tokenizer.

This prepares the dataset in tokenized form, ready for continual pretraining using an autoregressive language modeling objective.


In [15]:
# Load the tokenizer from the base model (Mistral-7B-Instruct)
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    use_fast=False,                 # Use the slow tokenizer (more flexible/customizable)
    trust_remote_code=True,        # Trust custom tokenizer code from remote repo (required by Mistral)
    add_eos_token=True             # Automatically add EOS token at the end of sequences
)

# If the tokenizer doesn't have a pad token, use the EOS token instead
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": tokenizer.eos_token})

# Tokenize the entire dataset
tokenized_train_dataset = []

# Loop through each text chunk and tokenize it
for phrase in train_dataset:
    tokenized_train_dataset.append(tokenizer(phrase["text"]))


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
tokenized_train_dataset[56]

{'input_ids': [1, 1164, 3597, 1501, 3910, 2091, 1971, 11054, 8798, 5477, 8509, 1072, 1246, 11909, 5010, 29481, 1066, 1800, 1281, 1504, 1171, 3192, 1246, 1597, 2068, 1610, 1036, 1103, 4433, 5326, 1032, 2042, 1070, 3723, 1137, 12516, 21594, 1040, 8525, 1070, 6873, 2756, 1245, 7055, 1032, 24166, 1330, 1603, 1245, 6548, 2027, 1158, 2282, 1767, 29477, 1968, 2487, 1709, 1077, 1159, 1968, 6893, 29266, 1968, 1454, 2147, 1283, 1072, 1454, 6499, 1066, 1420, 12482, 1968, 1330, 1504, 1171, 2396, 1137, 1597, 1115, 2971, 2110, 2712, 1066, 6799, 1040, 7256, 2242, 1070, 1673, 1461, 1065, 1040, 5493, 29501, 29508, 29542, 29551, 29502, 29481, 3542, 1066, 3495, 8714, 1072, 1970, 1610, 1040, 6270, 1070, 1164, 6413, 6482, 3667, 1631, 1227, 12635, 1806, 1040, 6482, 4386, 1065, 1229, 2139, 1598, 1042, 6235, 1448, 7116, 1065, 5936, 1452, 1567, 1968, 2611, 1155, 6482, 1968, 3304, 1125, 5253, 1072, 15683, 1968, 4313, 1968, 1158, 1117, 15653, 1245, 1036, 1103, 4433, 1036, 3476, 1070, 1032, 4019, 1066, 1229, 2139

## 🧪 Enable Gradient Checkpointing and Apply LoRA for PEFT

This section prepares the Mistral-7B model for **efficient continual pretraining** by:

### 🔄 1. Gradient Checkpointing
- **Purpose**: Reduces memory usage during training by trading compute for memory.
- **Effect**: Intermediate activations are recomputed during backpropagation instead of being stored.

### 🔧 2. LoRA (Low-Rank Adaptation)
Applies LoRA configuration via the `peft` library to enable **parameter-efficient training**:
- Only a small set of parameters are trained, while the rest of the model is frozen.
- Ideal for large models when full fine-tuning is too costly.

### 📌 LoRA Config Explanation:
- `r=8`: Rank of the low-rank decomposition (smaller = more efficient).
- `lora_alpha=64`: Scaling factor for LoRA updates.
- `target_modules`: Specifies which transformer submodules to modify (projection layers).
- `bias="none"`: Only LoRA adapters are trained, not biases.
- `lora_dropout=0.05`: Dropout to help generalize the low-rank updates.
- `task_type="CAUSAL_LM"`: Specifies that this is for a causal language modeling task (autoregressive generation).

This configuration ensures training efficiency without sacrificing much performance.


In [17]:
# Enable gradient checkpointing to reduce memory usage during training
model.gradient_checkpointing_enable()

# Prepare the model for 4-bit training using PEFT (required before applying LoRA)
model = prepare_model_for_kbit_training(model)

# Configure LoRA: low-rank adaptation setup for efficient fine-tuning
config = LoraConfig(
    r=8,  # Rank for low-rank decomposition (smaller rank = more efficient, but lower capacity)
    lora_alpha=64,  # Scaling factor applied to the LoRA updates
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # Target linear layers in transformer
    bias="none",  # Do not fine-tune biases
    lora_dropout=0.05,  # Dropout rate to apply within LoRA layers
    task_type="CAUSAL_LM"  # Task type: causal language modeling (next-token prediction)
)

# Apply the LoRA configuration to the model using PEFT
model = get_peft_model(model, config)


## 🚀 Launch Continual Pretraining with Transformers Trainer

This section sets up and launches the training process using Hugging Face’s `Trainer` API.

### 🧠 Model Training Setup:
- **Model**: LoRA-adapted Mistral-7B-Instruct with 4-bit quantization.
- **Trainer API**: Manages training loop, gradient accumulation, logging, and saving.

### ⚙️ TrainingArguments Highlights:
- `output_dir`: Directory to save checkpoints and final model.
- `per_device_train_batch_size=2`: Small batch size due to large model size and memory constraints.
- `gradient_accumulation_steps=2`: Accumulates gradients to simulate larger batch size.
- `num_train_epochs=10`: Number of epochs to train.
- `learning_rate=1e-4`: Learning rate for fine-tuning.
- `optim="paged_adamw_8bit"`: Memory-efficient optimizer from bitsandbytes.
- `bf16=False`: No bfloat16 used here (can be enabled if supported by hardware).
- `save_strategy="epoch"`: Save model at the end of each epoch.
- `save_steps=50`: Save checkpoint every 50 steps (redundant here due to `save_strategy`, but harmless).
- `logging_steps=50`: Log metrics every 50 steps.
- `logging_dir="./log"`: Directory to store training logs.

### 🧱 Data Collator:
- Uses `DataCollatorForLanguageModeling` with `mlm=False`, since this is **causal language modeling** (next-token prediction), not masked language modeling.

Lastly, `model.config.use_cache` is disabled to avoid caching overhead, which interferes with gradient checkpointing.


In [18]:
# Initialize Hugging Face Trainer with model, dataset, training arguments, and data collator
trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,

    # Define training hyperparameters and behaviors
    args=transformers.TrainingArguments(
        output_dir="./finetunedModel",           # Directory to save model checkpoints
        per_device_train_batch_size=2,           # Batch size per GPU
        gradient_accumulation_steps=2,           # Accumulate gradients to simulate larger batch size
        num_train_epochs=1,                     # Total number of training epochs
        learning_rate=1e-4,                      # Learning rate for optimizer
        #max_steps=10,                        # Optional: train for fixed number of steps
        bf16=False,                              # Use bfloat16 if supported (False here)
        optim="paged_adamw_8bit",                # Use 8-bit AdamW optimizer from bitsandbytes
        logging_dir="./log",                     # Where to store training logs
        save_strategy="epoch",                   # Save model checkpoint at end of each epoch
        save_steps=50,                           # (Redundant if save_strategy="epoch") Save every 50 steps
        logging_steps=50                        # Log training metrics every 50 steps
    ),

    # Causal language modeling requires next-token prediction (not MLM)
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

# Disable cache usage (necessary for gradient checkpointing to work properly)
model.config.use_cache = False

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [19]:
# Start training
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,1.917200
100,1.895500
150,1.856800
200,1.902400
250,1.816100
300,1.847700
350,1.885600
400,1.851000
450,1.884200
500,1.837200


TrainOutput(global_step=1250, training_loss=1.8475761840820313, metrics={'train_runtime': 25201.3679, 'train_samples_per_second': 0.198, 'train_steps_per_second': 0.05, 'total_flos': 1.1793247560464794e+17, 'train_loss': 1.8475761840820313, 'epoch': 1.0})

## 🧪 Inference & Testing the Continually Pretrained Model

This section runs **inference** using the continually pretrained Mistral-7B-Instruct model to test its response on a telecom-specific query.

### 🧠 Prompt Construction:
- The input prompt is phrased to elicit a concise and accurate answer:



### ⚙️ Inference Process:
- The prompt is tokenized and moved to GPU.
- The model is set to evaluation mode (`model.eval()`).
- Inference is performed using `generate()` with a token limit of 1024.
- Output is decoded and printed without special tokens.
- GPU memory is cleared with `torch.cuda.empty_cache()` after each run.

> **Note**: This test function was executed **multiple times** using different telecom-related questions to evaluate how well the model responded after continual pretraining. The goal was to observe improvements in domain-specific reasoning and response fluency.

This evaluation approach helps validate the effectiveness of continual pretraining before moving on to instruction fine-tuning.


In [20]:
user_question = "when the concentration of gentamicin was doubled"

eval_prompt = f"Just answer this question accurately and concisely.\nQuestion: {user_question} "

promptTokenized = tokenizer(user_question, return_tensors="pt").to("cuda")

model.eval()

with torch.no_grad():
  print(tokenizer.decode(model.generate(**promptTokenized, max_new_tokens=1024)[0], skip_special_tokens=True))
  torch.cuda.empty_cache()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


when the concentration of gentamicin was doubled 1 . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentamicin was doubled . the concentration of gentam

In [21]:
user_question = "participants were instructed to refrain from smoking , consuming any food and beverages except water , and"

promptTokenized = tokenizer(user_question, return_tensors="pt").to("cuda")

model.eval()

with torch.no_grad():
  print(tokenizer.decode(model.generate(**promptTokenized, max_new_tokens=1024)[0], skip_special_tokens=True))
  torch.cuda.empty_cache()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


participants were instructed to refrain from smoking , consuming any food and beverages except water , and 1 . the study was conducted in a university hospital in the city of tehran , iran . the study population consisted of 100 patients with type 2 diabetes mellitus who were referred to the endocrinology clinic of the hospital . the inclusion criteria were as follows : patients with type 2 diabetes mellitus , aged 1865 years , and with a body mass index ( bmi ) of 25 kg / m . the exclusion criteria were as follows : patients with a history of cardiovascular disease , renal failure , liver disease , or malignancy , and those who were taking medications that could affect blood pressure , such as calcium channel blockers , angiotensin - converting enzyme inhibitors , or angiotensin receptor blockers . the study protocol was approved by the ethics committee of the university hospital , and all participants provided written informed consent . the study was conducted in a university hospita

In [22]:
user_question = "The glycocalyx also modulates the inflammatory response by preventing leukocyte adhesion and binding numerous ligands"

promptTokenized = tokenizer(user_question, return_tensors="pt").to("cuda")

model.eval()

with torch.no_grad():
  print(tokenizer.decode(model.generate(**promptTokenized, max_new_tokens=1024)[0], skip_special_tokens=True))
  torch.cuda.empty_cache()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


The glycocalyx also modulates the inflammatory response by preventing leukocyte adhesion and binding numerous ligands 1 . the glycocalyx is a thin layer of proteoglycans and glycoproteins that covers the endothelial surface . it is a dynamic structure that is constantly being remodeled and is essential for the maintenance of endothelial barrier function . the glycocalyx is composed of a core protein , a proteoglycan , and a glycosaminoglycan ( gag ) chain . the core protein is a large protein that is attached to the endothelial cell surface by a glycosylphosphatidylinositol ( gpi ) anchor . the proteoglycan is a large protein that is attached to the core protein by a disulfide bond . the gag chain is a long chain of repeating disaccharide units that is attached to the proteoglycan by a covalent bond . the glycocalyx is a dynamic structure that is constantly being remodeled and is essential for the maintenance of endothelial barrier function . the glycocalyx is composed of a core protei

## ☁️ Merge LoRA Weights and Push Model to Hugging Face Hub

After continual pretraining, the **LoRA-adapted model** is merged back into the base Mistral-7B-Instruct weights for easier deployment and downstream fine-tuning.

### 🧩 What Happens Here:

- `model.merge_and_unload()`:
  - **Merges** the trained LoRA adapters into the base model weights.
  - **Unloads** the adapter-specific structure (making the model standard again).
  - Resulting model behaves like a fully fine-tuned Mistral-7B variant.

- `push_to_hub(new_model)`:
  - Uploads the final merged model and tokenizer to the Hugging Face Hub under the name:
    ```
    Cintinually-Pre-trained-Mistral-7B
    ```
    *(Note: Typo in “Continually” retained as per original code.)*

This version can now be **easily downloaded** later for further instruction fine-tuning using prompt-based datasets.

✅ This completes the continual pretraining phase.


In [23]:
model = model.merge_and_unload()  # This fuses the learned LoRA weights into the base model

/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [24]:
new_model = "MedConnect-Cintinually-Pre-trained-Mistral-7B"
model.push_to_hub(new_model) # Online saving
tokenizer.push_to_hub(new_model) # Online saving

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Agaba-Embedded4/MedConnect-Cintinually-Pre-trained-Mistral-7B/commit/e245feab4c090ed6f2baaf380a898362c4b2daa5', commit_message='Upload tokenizer', commit_description='', oid='e245feab4c090ed6f2baaf380a898362c4b2daa5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Agaba-Embedded4/MedConnect-Cintinually-Pre-trained-Mistral-7B', endpoint='https://huggingface.co', repo_type='model', repo_id='Agaba-Embedded4/MedConnect-Cintinually-Pre-trained-Mistral-7B'), pr_revision=None, pr_num=None)

In [25]:
base_model_id = "Agaba-Embedded4/MedConnect-Cintinually-Pre-trained-Mistral-7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16

)

model = AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config = bnb_config)

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/quantizers/auto.py:222: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/4.68G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [26]:
tokenizer = AutoTokenizer.from_pretrained("Agaba-Embedded4/MedConnect-Cintinually-Pre-trained-Mistral-7B", use_fast=False, trust_remote_code = True, add_eos_token = True)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [27]:
user_question = "in addition to food items , the ffq included questions about type and duration of vitamin / mineral supplement use?"

promptTokenized = tokenizer(user_question, return_tensors="pt").to("cuda")

model.eval()

with torch.no_grad():
  print(tokenizer.decode(model.generate(**promptTokenized, max_new_tokens=1024)[0], skip_special_tokens=True))
  torch.cuda.empty_cache()

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


in addition to food items , the ffq included questions about type and duration of vitamin / mineral supplement use? 1. how many days per week do you usually eat fish ? ( 0 = never , 1 = 1 day per week , 2 = 2 days per week , 3 = 3 days per week , 4 = 4 days per week , 5 = 5 days per week , 6 = 6 days per week , 7 = 7 days per week ) ? 2. how many servings of fish do you usually eat per day ? ( 1 = 1 serving , 2 = 2 servings , 3 = 3 servings , 4 = 4 servings , 5 = 5 servings ) ? 3. how many days per week do you usually eat poultry ? ( 0 = never , 1 = 1 day per week , 2 = 2 days per week , 3 = 3 days per week , 4 = 4 days per week , 5 = 5 days per week , 6 = 6 days per week , 7 = 7 days per week ) ? 4. how many servings of poultry do you usually eat per day ? ( 1 = 1 serving , 2 = 2 servings , 3 = 3 servings , 4 = 4 servings , 5 = 5 servings ) ? 5. how many days per week do you usually eat red meat ? ( 0 = never , 1 = 1 day per week , 2 = 2 days per week , 3 = 3 days per week , 4 = 4 day